# Clase 9: Bases de datos relacionales y SQL

**Objetivos de aprendizaje:**
- Explicar el modelo de tablas relacionadas.
- Describir las garantías ACID.
- Escribir consultas SQL básicas (`SELECT`, `WHERE`, `JOIN`) usando `sqlite3`.

**Duración estimada:** 120 min


## Para arrancar: ¿qué es una base de datos relacional?

Antes de tocar código, pensemos en algo que ya conocéis: una hoja de cálculo con varias pestañas. Una pestaña de clientes, otra de transacciones. Si queréis saber cuánto ha gastado un cliente concreto, tenéis que ir a la pestaña de transacciones, buscar las filas de ese cliente y sumarlas a mano.

Una **base de datos relacional** hace justo eso, pero de forma automática y a cualquier escala. Los datos viven en **tablas** (filas = registros, columnas = atributos), y las tablas se **relacionan** entre sí mediante un campo compartido — normalmente un identificador. El software que gestiona todo esto se llama **DBMS** (Database Management System).

Hoy vamos a construir, en vivo, una mini base de datos con dos tablas relacionadas y le vamos a hacer preguntas reales en SQL.

In [ ]:
import sqlite3
import pandas as pd

# ':memory:' crea la base de datos directamente en RAM: nada de
# archivos externos, perfecto para clase y para Colab.
conexion = sqlite3.connect(':memory:')
cursor = conexion.cursor()

# Tabla 1: clientes
cursor.execute('''
    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        ciudad TEXT
    )
''')

# Tabla 2: transacciones, relacionada con clientes por id_cliente
cursor.execute('''
    CREATE TABLE transacciones (
        id_transaccion INTEGER PRIMARY KEY,
        id_cliente INTEGER,
        importe REAL NOT NULL,
        FOREIGN KEY (id_cliente) REFERENCES clientes (id_cliente)
    )
''')

print("Tablas creadas.")

In [ ]:
# Poblamos las tablas con executemany: mucho más eficiente que
# hacer un INSERT por fila cuando tenemos varias a la vez.
clientes_nuevos = [
    (1, 'Ana Torres', 'Madrid'),
    (2, 'Luis Peña', 'Bogotá'),
    (3, 'Carla Ruiz', 'Madrid'),
    (4, 'Diego Soto', 'CDMX'),
]
cursor.executemany(
    'INSERT INTO clientes (id_cliente, nombre, ciudad) VALUES (?, ?, ?)',
    clientes_nuevos
)

transacciones_nuevas = [
    (101, 1, 59.90),
    (102, 1, 120.00),
    (103, 2, 34.50),
    (104, 3, 89.99),
    (105, 3, 15.00),
    (106, 4, 200.00),
]
cursor.executemany(
    'INSERT INTO transacciones (id_transaccion, id_cliente, importe) VALUES (?, ?, ?)',
    transacciones_nuevas
)

conexion.commit()
print("Datos insertados.")

## Antes de consultar: las garantías ACID

Cuando hicimos `conexion.commit()` en la celda anterior, confiamos en que la base de datos guardó los cambios de forma fiable. Eso es posible gracias a las garantías **ACID**, el pilar de las bases de datos relacionales:

- **A**tomicidad: una operación se ejecuta entera o no se ejecuta.
- **C**onsistencia: la base de datos pasa de un estado válido a otro estado válido, nunca a medias.
- **A**islamiento: dos operaciones simultáneas no se pisan entre sí.
- **D**urabilidad: una vez confirmado un cambio, sobrevive aunque el sistema falle justo después.

Ahora sí, vamos a preguntarle cosas a nuestra base de datos.

In [ ]:
# WHERE: filtramos filas según una condición
resultado_where = pd.read_sql_query(
    "SELECT nombre, ciudad FROM clientes WHERE ciudad = 'Madrid'",
    conexion
)
print("Clientes de Madrid:")
print(resultado_where)

In [ ]:
# JOIN: cruzamos las dos tablas por el campo que comparten
resultado_join = pd.read_sql_query('''
    SELECT clientes.nombre, transacciones.importe
    FROM transacciones
    JOIN clientes ON transacciones.id_cliente = clientes.id_cliente
''', conexion)
print("Transacciones con el nombre del cliente:")
print(resultado_join)

In [ ]:
# GROUP BY: agrupamos filas y calculamos un agregado por grupo
resultado_group = pd.read_sql_query('''
    SELECT clientes.nombre, SUM(transacciones.importe) AS gasto_total
    FROM transacciones
    JOIN clientes ON transacciones.id_cliente = clientes.id_cliente
    GROUP BY clientes.nombre
    ORDER BY gasto_total DESC
''', conexion)
print("Gasto total por cliente:")
print(resultado_group)

## Lo que acabamos de ver, en una frase

Con `WHERE` filtramos, con `JOIN` cruzamos tablas relacionadas y con `GROUP BY` agregamos por grupo. Fijaos en que nunca escribimos un bucle: le describimos a SQL *qué* queríamos, y el motor decidió *cómo* conseguirlo.

Esto es solo posible porque diseñamos bien la relación entre `clientes` y `transacciones` desde el principio, usando `id_cliente` como clave compartida (clave foránea) — exactamente el modelo de tabla relacional del que hablamos al empezar la clase.

Motores reales que usan este mismo lenguaje SQL: **IBM Db2, SQL Server, MySQL, Oracle, PostgreSQL** (y sus versiones "as a service" en la nube). Los usos más comunes son sistemas **OLTP** (transacciones del día a día) y **data warehousing/OLAP** (análisis histórico) — de esto último hablaremos en la clase 11.

In [ ]:
conexion.close()

## Resumen de la clase

- Una tabla relacional tiene filas (registros) y columnas (atributos); se relaciona con otras tablas mediante una clave compartida.
- Las bases de datos relacionales garantizan ACID: Atomicidad, Consistencia, Aislamiento, Durabilidad.
- `sqlite3` nos permite crear y consultar una base de datos relacional real en memoria, sin instalar nada.
- `SELECT ... WHERE` filtra, `JOIN` cruza tablas relacionadas, `GROUP BY` agrupa y agrega.
- `pandas.read_sql_query` es la forma más cómoda de traer el resultado de una consulta SQL como DataFrame.

**Capítulo del manual:** [11 — Bases de datos relacionales y SQL](../manual/11_bases_relacionales_sql.md)

**Notebook de práctica:** [clase09_practica_sql_bases_relacionales.ipynb](../notebooks_practica/clase09_practica_sql_bases_relacionales.ipynb)